In [ ]:
import os
from random import random

import custom_dataset as dataset
import helper_functions as hf
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
import torch.nn as nn
import segmentation_models_pytorch as smp
import torch
from tqdm.auto import tqdm
from model import InVolcModel


In [ ]:
device = hf.set_device()
seed = 42
hf.set_seed(seed)


In [ ]:
train_transforms = transforms.Compose([
    # transforms.Resize((128, 128)),
    transforms.ToTensor(),
])

test_transforms = transforms.Compose([
    # transforms.Resize((128, 128)),
    transforms.ToTensor(),
])

In [ ]:
train_data = dataset.InVolcDataset(
    metadata_csv='metadata_sample_01.csv',
    transform=train_transforms,
    subset='train')
test_data = dataset.InVolcDataset(
    metadata_csv='metadata_sample_01.csv',
    transform=test_transforms,
    subset='test')

In [ ]:
train_dataloader = DataLoader(
    train_data,
    batch_size=8,
    shuffle=True,
    num_workers=4
)
test_dataloader = DataLoader(
    test_data,
    batch_size=5,
    shuffle=False,
    num_workers=4
)


In [ ]:
# python
class InVolcModel(nn.Module):
    def __init__(self,
                 encoder_name: str = 'resnet34',
                 encoder_weights: str = 'imagenet',
                 in_channels: int = 3,
                 classes_seg: int = 1,
                 num_classes_clf: int = 1):
        super().__init__()
        self.unet = smp.Unet(
            encoder_name=encoder_name,
            encoder_weights=encoder_weights,
            in_channels=in_channels,
            classes=classes_seg,
        )

        self.classification_head = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(start_dim=1),
            nn.Linear(self.unet.encoder.out_channels[-1], num_classes_clf)
        )

    def forward(self, x):
        """
        x: (B, in_channels, H, W)
        returns:
            clf_logits: (B, num_classes_clf)
            seg_logits: (B, classes_seg, H, W)
        """
        features = self.unet.encoder(x)
        bottleneck = features[-1]

        # Pasar la lista/tupla completa al decoder (sin usar '*')
        decoder_output = self.unet.decoder(features)
        seg_logits = self.unet.segmentation_head(decoder_output)

        clf_logits = self.classification_head(bottleneck)

        return clf_logits, seg_logits

In [ ]:
model = InVolcModel(
        encoder_name="resnet34",
        encoder_weights="imagenet",
        in_channels=1,
        classes_seg=1,
        num_classes_clf=1)
x = torch.randn(2, 1, 256, 256)  # batch de 2 imágenes 1x256x256
clf_logits, seg_logits = model(x)
print("clf_logits:", clf_logits.shape)    # esperado: (2, 1)
print("seg_logits:", seg_logits.shape)    # esperado: (2, 1, 256, 256)


In [ ]:
#visualizar el shape de una mask
muestra = train_data[42]
mask = muestra['mask']
print("mask shape:", mask.shape)  # esperado: (H, W) o (

In [ ]:
for batch in train_dataloader:
    data, masks, labels = hf.prepare_batch(
        batch,
        device=device,
        normalize_mask=True,
        require_float_labels=True
    )
    # solo el primer batch para debug
    break


In [ ]:
class DiceBCELoss(nn.Module):
    def __init__(self, bce_weight=0.5, dice_weight=0.5):
        super().__init__()
        self.bce_weight = bce_weight
        self.dice_weight = dice_weight
        self.bce = nn.BCEWithLogitsLoss()
        self.dice = smp.losses.DiceLoss(mode='binary')

    def forward(self, preds, targets):
        bce_loss = self.bce(preds, targets)
        dice_loss = self.dice(preds, targets)
        return self.bce_weight * bce_loss + self.dice_weight * dice_loss

In [ ]:
loss_fn_clf = nn.BCEWithLogitsLoss()
loss_fn_seg = DiceBCELoss(bce_weight=0.7, dice_weight=0.2)

In [ ]:
from metrics import MetricsManager

metrics = MetricsManager(device)

In [ ]:
def train_step(model: torch.nn.Module,
               dataloader: DataLoader,
               loss_fn_clf: torch.nn.Module,
               loss_fn_seg: torch.nn.Module,
               optimizer: torch.optim.Optimizer,
               device: torch.device=device
               ):

    clf_losses, seg_losses = [], []
    model.train()
    loss = 0.0

    for idx, batch in enumerate(dataloader):

        images, masks, labels = hf.prepare_batch(
            batch, device=device,
            normalize_mask=True,
            require_float_labels=True
        )

        # forward pass
        clf_logits, seg_logits = model(images)

        # compute losses
        loss_clf = loss_fn_clf(clf_logits, labels.unsqueeze(1))
        loss_seg = loss_fn_seg(seg_logits, masks)

        loss = loss_clf + 2*loss_seg

        # backward pass and optimization
        optimizer.zero_grad()
        loss.backward()

        # opcional pero muy recomendable
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
        optimizer.step()

        # logging
        clf_losses.append(loss_clf.item())
        seg_losses.append(loss_seg.item())

        # clf_probs = torch.sigmoid(clf_logits)        # (B,1)
        # clf_preds = (clf_probs > 0.5).float()
        #
        # seg_probs = torch.sigmoid(seg_logits)        # (B,1,H,W)
        # seg_preds = (seg_probs > 0.5).float()# (B,1)

    mean_clf = sum(clf_losses) / len(clf_losses)
    mean_seg = sum(seg_losses) / len(seg_losses)
    mean_tot = mean_clf + 2*mean_seg

    return mean_tot, mean_clf, mean_seg

In [ ]:
def test_step(model,
              dataloader,
              loss_fn_clf,
              loss_fn_seg,
              device):

    model.eval()
    metrics.reset()
    clf_losses, seg_losses = [], []

    all_clf_preds = []
    all_clf_targets = []

    all_seg_preds = []
    all_seg_targets = []

    with torch.no_grad():
        for batch in dataloader:

            images, masks, labels = hf.prepare_batch(
                batch, device=device,
                normalize_mask=True,
                require_float_labels=True
            )

            # Forward
            clf_logits, seg_logits = model(images)

            # Losses
            loss_clf = loss_fn_clf(clf_logits, labels.unsqueeze(1))
            loss_seg = loss_fn_seg(seg_logits, masks)

            clf_losses.append(loss_clf.item())
            seg_losses.append(loss_seg.item())



            # Predictions
            clf_probs = torch.sigmoid(clf_logits)
            clf_preds = (clf_probs > 0.5).float()

            metrics.update(clf_preds.squeeze(1),labels)

            seg_probs = torch.sigmoid(seg_logits)
            seg_preds = (seg_probs > 0.5).float()

            # Save for computing metrics outside
            all_clf_preds.append(clf_preds.cpu())
            all_clf_targets.append(labels.cpu())

            all_seg_preds.append(seg_preds.cpu())
            all_seg_targets.append(masks.cpu())

    # Concatenate everything
    all_clf_preds = torch.cat(all_clf_preds)
    all_clf_targets = torch.cat(all_clf_targets)

    all_seg_preds = torch.cat(all_seg_preds)
    all_seg_targets = torch.cat(all_seg_targets)



    result = {
        "clf_loss": sum(clf_losses) / len(clf_losses),
        "seg_loss": sum(seg_losses) / len(seg_losses),
        "total_loss": sum(clf_losses) / len(clf_losses) + 2 * sum(seg_losses) / len(seg_losses),

        # Raw predictions → for computing metrics outside
        "clf_preds": all_clf_preds,
        "clf_targets": all_clf_targets,
        "seg_preds": all_seg_preds,
        "seg_targets": all_seg_targets,
    }

    return result

In [ ]:
def train(model: torch.nn.Module,
          train_dataloader: DataLoader,
          test_dataloader: DataLoader,
          loss_fn_clf: torch.nn.Module,
          loss_fn_seg: torch.nn.Module,
          optimizer: torch.optim.Optimizer,
          num_epochs: int = 10,
          device: torch.device = device):

    test_results = []
    for epoch in tqdm(range(num_epochs)):
        train_total_loss, train_clf_loss, train_seg_loss = train_step(
            model,
            train_dataloader,
            loss_fn_clf,
            loss_fn_seg,
            optimizer,
            device
        )

        test_results_step = test_step(
            model,
            test_dataloader,
            loss_fn_clf,
            loss_fn_seg,
            device
        )

        print(f"Epoch {epoch+1}/{num_epochs}")
        print(f"  Train Loss: {train_total_loss:.4f} (Clf: {train_clf_loss:.4f}, Seg: {train_seg_loss:.4f})")
        print(f"  Test Loss: {test_results_step['total_loss']:.4f} (Clf: {test_results_step['clf_loss']:.4f}, Seg: {test_results_step['seg_loss']:.4f})")
        test_results.append(test_results_step)

    return test_results

In [ ]:
hf.set_seed(seed)
invol_v1 = InVolcModel(
        encoder_name="resnet34",
        encoder_weights="imagenet",
        in_channels=3,
        classes_seg=1,
        num_classes_clf=1)
invol_v1.to(device)
optimizer = torch.optim.Adam(invol_v1.parameters(), lr=1e-4)
result = train(
    invol_v1,
    train_dataloader,
    test_dataloader,
    loss_fn_clf,
    loss_fn_seg,
    optimizer,
    num_epochs=10,
    device=device
)


In [ ]:
metrics.compute()

In [ ]:
import matplotlib.pyplot as plt

# Suponiendo que ya hiciste:
# result = train(...)

# Extraer pérdidas de test por epoch
test_total_loss = [r["total_loss"] for r in result]
test_clf_loss   = [r["clf_loss"]   for r in result]
test_seg_loss   = [r["seg_loss"]   for r in result]

epochs = range(1, len(result) + 1)

plt.figure(figsize=(8, 5))
plt.plot(epochs, test_total_loss, label="Test total loss")
plt.plot(epochs, test_clf_loss,   label="Test clf loss")
plt.plot(epochs, test_seg_loss,   label="Test seg loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Evolución de la función de pérdida (test)")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
def make_predictions(model: torch.nn.Module,
                     data: list,
                     device: torch.device = device):
    model.eval()
    model.to(device)
    all_clf_preds = []
    all_seg_preds = []


    with torch.inference_mode():
        for i in data:
            tensor = hf.preprocess_image(i)

            clf_logits, seg_logits = model(tensor)

            clf_probs = torch.sigmoid(clf_logits)
            clf_preds = (clf_probs > 0.5).float()
            all_clf_preds.append(clf_preds.cpu())

            seg_probs = torch.sigmoid(seg_logits)
            seg_preds = (seg_probs > 0.5).float()
            all_seg_preds.append(seg_preds.cpu())

In [ ]:
import pandas as pd
from math import ceil

def extract_balanced_sample(csv_path: str,
                            output_path: str,
                            sample_size: int = 100,
                            min_pos_fraction: float = 0.30,
                            random_state: int | None = 42) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    if 'label_bin' not in df.columns:
        raise ValueError("La columna 'label_bin' es obligatoria.")

    required_pos = ceil(sample_size * min_pos_fraction)
    positives = df[df['label_bin'] == 1]
    negatives = df[df['label_bin'] == 0]

    if len(df) < sample_size:
        raise ValueError(f"Solo hay {len(df)} filas; se requieren {sample_size}.")
    if len(positives) < required_pos:
        raise ValueError(f"No hay suficientes positivos: {len(positives)} disponibles, {required_pos} requeridos.")

    sampled_pos = positives.sample(n=required_pos, replace=False, random_state=random_state)
    sampled_neg = negatives.sample(n=sample_size - required_pos,
                                   replace=False,
                                   random_state=None if random_state is None else random_state + 1)

    sample_df = pd.concat([sampled_pos, sampled_neg]).sample(frac=1.0, random_state=random_state).reset_index(drop=True)
    sample_df.to_csv(output_path, index=False)
    return sample_df

In [ ]:
data = "../data/metadata.csv"
out = extract_balanced_sample(
        csv_path=data,
        output_path="metadata_sample_02.csv",
        sample_size=115,
        min_pos_fraction=0.3,
        random_state=42)


In [ ]:
# guardar el archivo our
out

In [ ]:
hp.split_metadata(
    metadata_csv='metadata_sample_02.csv',
    train_csv='metadata_sample_02_train.csv',
    test_csv='metadata_sample_02_test.csv',
    train_fraction=0.8,
    random_state=42
)